In [ ]:
from essential.data import load_regulondb_full
import scanpy as sc
from cellbox import CellBoxEstimator

ref_db = load_regulondb_full()

adata_path = "/workspace/data/de122_lce75/adata_de122_lce75_merged.h5ad"
adata = sc.read_h5ad(adata_path)
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

adata = adata[adata.obs["target"].notna()].copy()

In [ ]:
import numpy as np
import pandas as pd

var_lower = adata.var_names.str.lower()
name_to_idx = pd.Series(np.arange(len(var_lower)), index=var_lower)

Amask = np.zeros((len(adata.var_names), len(adata.var_names)), dtype=np.float32)

pairs = ref_db[["target_gene", "regulator_gene"]].copy()
pairs["t_idx"] = pairs["target_gene"].map(name_to_idx)
pairs["r_idx"] = pairs["regulator_gene"].map(name_to_idx)
pairs = pairs.dropna(subset=["t_idx", "r_idx"])

Amask[pairs["t_idx"].astype(int).values, pairs["r_idx"].astype(int).values] = 1.0

In [ ]:
(Amask.sum(-1) == 0).sum()

In [ ]:
ref_db["regulator_gene"].unique()

In [ ]:
adata.var_names.str.lower()

In [ ]:
est = CellBoxEstimator(
    adata,
    perturbation_col="target",   # your adata's column name
    control_key="nontargeting",
)

est.fit(
    learning_rate=1e-3,
    n_epochs=10_000,
    batch_size=8_000,
    train_size=0.9,
    early_stopping_patience=2_000,
    early_stopping_metric="reco_loss",
    log_every_n_epochs=500,
)
